In [1]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd


In [9]:
device = "cuda"

In [10]:
df = pd.read_csv('../Data/raw/tcga_simple_train.csv')

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=42)

In [13]:
# cargamos el tokenizer y el modelo preentrenado de BERT
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

# modo evaluación - no queremos entrenar, queremos obtener los embeddings
model.eval()

# movemos el modelo a la GPU
model.to(device)

# función para obtener los embeddings de un texto
def get_token_embeddings(text, max_length=256):
    tokens = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=max_length) #tokenizamos el text y lo convertimos a tensores de PyTorch
    tokens = {key: val.to(device) for key, val in tokens.items()}

    with torch.no_grad(): #no queremos calcular gradientes, solo obtener los embeddings
        outputs = model(**tokens) #obtenemos los outputs del modelo

    embeddigns = outputs.last_hidden_state.cpu().numpy() #obtenemos los embeddings de la última capa oculta y los convertimos a numpy
    return embeddigns[:, 1:-1, :] #eliminamos los tokens especiales [CLS] y [SEP]

'''
BERT devuelve un vector por token.
Pero para clasificar necesitamos un único vector por texto.
Para ello haremos Pooling sobre los vectores de los tokens.
'''
def bert_mean_embedding(text, max_length=256):
    embeddigns = get_token_embeddings(text, max_length)
    mean_vector = np.mean(embeddigns, axis=1) #hacemos pooling por media sobre los vectores de los tokens
    return mean_vector[0] #devolvemos el vector resultante

def bert_max_embedding(text, max_length=256):
    embeddigns = get_token_embeddings(text, max_length)
    max_vector = np.max(embeddigns, axis=1) #hacemos pooling por max sobre los vectores de los tokens
    return max_vector[0] #devolvemos el vector resultante


def bert_embedding(texts, pooling='mean', max_length=256):
    if pooling == 'mean':
        return np.array([bert_mean_embedding(text, max_length) for text in texts])
    elif pooling == 'max':
        return np.array([bert_max_embedding(text, max_length) for text in texts])
    else:
        raise ValueError("Pooling debe ser 'mean' o 'max'")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
X_train_bert = bert_embedding(X_train, 'max')
X_test_bert = bert_embedding(X_test, 'max')

KeyboardInterrupt: 

In [ ]:
import numpy as np
np.save("../Data/processed/X_train_bert_max.npy", X_train_bert)
np.save("../Data/processed/X_test_bert_max.npy", X_test_bert)